# Evaluating Retrieval-Augmented Generation

Before you can improve a RAG pipeline, you need to measure it. This notebook covers the main evaluation metrics for both the retrieval component (did we fetch the right documents?) and the generation component (is the answer correct?). You will implement several metrics from scratch, see where common metrics mislead, and build a small evaluation harness around a synthetic dataset.

In [ ]:
# Uncomment to install on Colab or a fresh environment
# !pip install rouge-score nltk bert-score pandas -q

import math
import re
import collections
from typing import List, Dict, Any, Tuple, Optional

import numpy as np
import pandas as pd

# We import these lazily in the relevant cells to avoid crashing if not installed
print("Core imports done.")

## 1a. Embedding Generation with `bge-large-en-v1.5`

Before evaluating retrieval, we need a way to turn text into dense vectors. `BAAI/bge-large-en-v1.5` is a strong English embedding model (1024-dimensional output) that consistently ranks near the top of the MTEB retrieval benchmark. "bge" stands for BAAI General Embedding.

Key details:
- Output dimension: 1024 (compared to 384 for bge-small).
- Normalization: always normalize embeddings before cosine or inner-product comparisons.
- Batch encoding: `model.encode(texts, batch_size=32)` processes multiple texts efficiently.

We will build a small document set and compute embeddings, then use them for all subsequent evaluation cells.

In [ ]:
# !pip install sentence-transformers faiss-cpu rank-bm25 scikit-learn seaborn -q

import numpy as np
from sentence_transformers import SentenceTransformer

EMBED_MODEL_ID = "BAAI/bge-large-en-v1.5"

# Small but topically diverse document set
DOCUMENTS = [
    {"id": "doc_0",  "title": "Photosynthesis",      "text": "Photosynthesis is the process by which plants use sunlight, water, and CO2 to produce glucose and oxygen in chloroplasts."},
    {"id": "doc_1",  "title": "DNA Replication",      "text": "DNA replication unwinds the double helix; each strand acts as a template for a new complementary strand, yielding two identical molecules."},
    {"id": "doc_2",  "title": "Newton's Laws",        "text": "Newton's second law states F=ma. His first law holds that objects remain at rest or in motion unless a net force acts on them."},
    {"id": "doc_3",  "title": "Rainbows",             "text": "Rainbows form when sunlight refracts, disperses, and internally reflects inside water droplets, spreading into a spectrum of colors."},
    {"id": "doc_4",  "title": "Pythagorean Theorem",  "text": "In a right triangle, the square of the hypotenuse equals the sum of squares of the other two sides: a^2 + b^2 = c^2."},
    {"id": "doc_5",  "title": "Machine Learning",     "text": "Machine learning is a subset of AI where systems learn from data to improve performance without being explicitly programmed for each task."},
    {"id": "doc_6",  "title": "Immune System",        "text": "The immune system fights viruses using innate immunity for rapid non-specific responses and adaptive immunity to produce antibodies and memory cells."},
    {"id": "doc_7",  "title": "Water Cycle",          "text": "The water cycle involves evaporation, condensation, precipitation, and collection, continuously moving water through Earth's systems."},
    {"id": "doc_8",  "title": "Inflation",            "text": "Inflation is the rate at which the general price level of goods and services rises over time, reducing the purchasing power of money."},
    {"id": "doc_9",  "title": "Quantum Entanglement", "text": "Quantum entanglement links particles so the state of one instantly constrains the other regardless of distance, defying classical intuition."},
    {"id": "doc_10", "title": "Gradient Descent",     "text": "Gradient descent is an optimization algorithm that iteratively adjusts model parameters in the direction of steepest loss reduction."},
    {"id": "doc_11", "title": "Transformer Model",    "text": "Transformers use self-attention to weigh relationships between all tokens simultaneously, enabling parallelization and long-range dependency capture."},
]

doc_texts = [d["text"] for d in DOCUMENTS]
doc_ids   = [d["id"]   for d in DOCUMENTS]

print(f"Loading embedding model: {EMBED_MODEL_ID} ...")
bge_model = SentenceTransformer(EMBED_MODEL_ID)

# encode with normalization so cosine similarity = dot product
embeddings = bge_model.encode(doc_texts, normalize_embeddings=True, show_progress_bar=True)

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"  Rows (documents): {embeddings.shape[0]}")
print(f"  Columns (embedding dim): {embeddings.shape[1]}")
print(f"\nFirst embedding L2 norm (should be 1.0): {np.linalg.norm(embeddings[0]):.6f}")

## 1b. Cosine Similarity Matrix Heatmap

Pairwise cosine similarity between all document embeddings reveals how the embedding space is structured. Documents on closely related topics should cluster together (high similarity), while unrelated documents should be far apart (low similarity).

With normalized embeddings, cosine similarity equals the dot product, so we can compute the full matrix with a single matrix multiplication: `sim = embeddings @ embeddings.T`.

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Pairwise cosine similarity (embeddings are already normalized, so dot product = cosine sim)
sim_matrix = embeddings @ embeddings.T   # shape: (12, 12)

titles = [d["title"] for d in DOCUMENTS]

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    sim_matrix,
    annot=True,
    fmt=".2f",
    xticklabels=titles,
    yticklabels=titles,
    cmap="Blues",
    vmin=0.0,
    vmax=1.0,
    linewidths=0.4,
    ax=ax,
)
ax.set_title("Pairwise Cosine Similarity (bge-large-en-v1.5)", fontsize=13)
plt.xticks(rotation=45, ha="right", fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

# Highlight the most and least similar pairs (excluding diagonal)
np.fill_diagonal(sim_matrix, 0)
max_idx = np.unravel_index(np.argmax(sim_matrix), sim_matrix.shape)
min_idx = np.unravel_index(np.argmin(sim_matrix), sim_matrix.shape)

print(f"\nMost similar pair:  '{titles[max_idx[0]]}' <-> '{titles[max_idx[1]]}' "
      f"(sim={sim_matrix[max_idx]:.4f})")
print(f"Least similar pair: '{titles[min_idx[0]]}' <-> '{titles[min_idx[1]]}' "
      f"(sim={sim_matrix[min_idx]:.4f})")

## 1c. FAISS Index: Build and Query

FAISS (Facebook AI Similarity Search) is a library for efficient similarity search over large collections of dense vectors. `IndexFlatIP` performs exact inner-product search, which equals cosine similarity when vectors are normalized. It is the right choice when the corpus is small (up to ~100k vectors) and you need perfect recall.

Steps:
1. Create an index with the embedding dimension.
2. `index.add()` the normalized corpus embeddings.
3. For a query, encode and normalize it, then call `index.search(query_vec, k)` which returns scores and indices for the top-k results.

In [ ]:
import faiss
import numpy as np

# Build the FAISS index
dim = embeddings.shape[1]   # 1024 for bge-large
faiss_index = faiss.IndexFlatIP(dim)
faiss_index.add(embeddings.astype(np.float32))
print(f"FAISS IndexFlatIP built. Vectors indexed: {faiss_index.ntotal}, dim={dim}")


def dense_retrieve(query: str, k: int = 5):
    """Retrieve top-k documents using dense embedding similarity."""
    query_vec = bge_model.encode([query], normalize_embeddings=True)
    scores, indices = faiss_index.search(query_vec.astype(np.float32), k)
    return [(doc_ids[i], DOCUMENTS[i]["title"], float(scores[0][rank]))
            for rank, i in enumerate(indices[0])]


# Test with several queries
test_queries_faiss = [
    "How do plants convert sunlight into energy?",
    "What optimization algorithm is used in deep learning?",
    "Explain the process of copying genetic material.",
]

for q in test_queries_faiss:
    results = dense_retrieve(q, k=5)
    print(f"\nQuery: {q}")
    print(f"{'Rank':<5} {'ID':<8} {'Score':>6}  Title")
    print("-" * 50)
    for rank, (doc_id, title, score) in enumerate(results, 1):
        print(f"  {rank:<4} {doc_id:<8} {score:>6.4f}  {title}")

## 2a. Precision@k and Recall@k: Formulas and Worked Examples

Two complementary metrics characterize how well the top-k results satisfy an information need:

**Precision@k** asks: "Of the k documents I returned, what fraction are actually relevant?"

```
Precision@k = |relevant_docs ∩ top_k_retrieved| / k
```

**Recall@k** asks: "Of all relevant documents that exist, what fraction did I find in my top k?"

```
Recall@k = |relevant_docs ∩ top_k_retrieved| / |relevant_docs|
```

The two metrics trade off against each other as k grows. Precision tends to fall as k increases (you pick up more irrelevant docs). Recall tends to rise (you find more of the relevant ones). The right choice of metric depends on your application: a recommendation system cares more about Precision (every result shown should be good), while a medical literature search cares more about Recall (missing a relevant paper could be costly).

In [ ]:
from typing import List
import matplotlib.pyplot as plt


def precision_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    """
    Precision@k: fraction of top-k retrieved docs that are relevant.

    Args:
        retrieved: Ordered list of retrieved doc IDs (best first).
        relevant:  Ground-truth set of relevant doc IDs.
        k:         Cutoff rank.
    Returns:
        Precision@k in [0, 1].
    """
    if k == 0:
        return 0.0
    top_k = set(retrieved[:k])
    relevant_set = set(relevant)
    return len(top_k & relevant_set) / k


def recall_at_k_v2(retrieved: List[str], relevant: List[str], k: int) -> float:
    """
    Recall@k: fraction of all relevant docs that appear in top-k results.

    Args:
        retrieved: Ordered list of retrieved doc IDs (best first).
        relevant:  Ground-truth set of relevant doc IDs.
        k:         Cutoff rank.
    Returns:
        Recall@k in [0, 1]. Returns 0 if relevant is empty.
    """
    if not relevant:
        return 0.0
    top_k = set(retrieved[:k])
    relevant_set = set(relevant)
    return len(top_k & relevant_set) / len(relevant_set)


# --- Worked example ---
# Suppose there are 3 relevant documents in a corpus of 10.
example_retrieved = ["doc_A", "doc_X", "doc_B", "doc_Y", "doc_C", "doc_Z", "doc_W", "doc_V", "doc_U", "doc_T"]
example_relevant  = ["doc_A", "doc_B", "doc_C"]

print("Worked example:")
print(f"Retrieved (in order): {example_retrieved}")
print(f"Relevant docs:        {example_relevant}\n")

k_values = [1, 2, 3, 5, 10]
print(f"{'k':<5} {'Precision@k':>12} {'Recall@k':>10}")
print("-" * 30)
for k in k_values:
    p = precision_at_k(example_retrieved, example_relevant, k)
    r = recall_at_k_v2(example_retrieved, example_relevant, k)
    print(f"{k:<5} {p:>12.3f} {r:>10.3f}")

# Plot precision-recall tradeoff as k increases
precisions = [precision_at_k(example_retrieved, example_relevant, k) for k in k_values]
recalls    = [recall_at_k_v2(example_retrieved, example_relevant, k) for k in k_values]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

ax1.plot(k_values, precisions, "bo-", label="Precision@k")
ax1.plot(k_values, recalls,    "rs-", label="Recall@k")
ax1.set_xlabel("k (cutoff rank)")
ax1.set_ylabel("Score")
ax1.set_title("Precision and Recall as k grows")
ax1.legend()
ax1.set_ylim(0, 1.05)

ax2.plot(recalls, precisions, "g^-")
for i, k in enumerate(k_values):
    ax2.annotate(f"k={k}", (recalls[i], precisions[i]), textcoords="offset points", xytext=(5, 5))
ax2.set_xlabel("Recall@k")
ax2.set_ylabel("Precision@k")
ax2.set_title("Precision-Recall Tradeoff")
ax2.set_xlim(0, 1.05)
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

## 2b. MRR: Mean Reciprocal Rank

MRR focuses specifically on *where* the first correct result appears. The formula for a set of Q queries is:

```
MRR = (1/|Q|) * sum_{i=1}^{|Q|} (1 / rank_i)
```

where `rank_i` is the rank of the first relevant document for query i. If no relevant document is retrieved, the contribution is 0.

MRR is:
- **1.0** if every query's first relevant document is at rank 1.
- **0.5** on average if first relevant documents typically appear at rank 2.
- **0.0** if no relevant documents are ever retrieved.

The worked test case below uses five queries with known retrieval orders to verify the implementation.

In [ ]:
from typing import List, Tuple


def reciprocal_rank_single(retrieved: List[str], relevant: List[str]) -> float:
    """1 / rank of first relevant doc, or 0 if none found."""
    relevant_set = set(relevant)
    for rank, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant_set:
            return 1.0 / rank
    return 0.0


def mean_reciprocal_rank_v2(query_results: List[Tuple[List[str], List[str]]]) -> float:
    """
    MRR = (1/|Q|) * sum_i (1 / rank_i)

    Args:
        query_results: list of (retrieved_doc_ids, relevant_doc_ids) per query.
    Returns:
        Mean Reciprocal Rank in [0, 1].
    """
    if not query_results:
        return 0.0
    rr_sum = sum(reciprocal_rank_single(ret, rel) for ret, rel in query_results)
    return rr_sum / len(query_results)


# --- Test case with known expected MRR ---
# Each tuple: (retrieved_list, relevant_list)
test_cases = [
    # Query 1: relevant at rank 1  -> RR = 1.0
    (["doc_A", "doc_B", "doc_C"], ["doc_A"]),
    # Query 2: relevant at rank 2  -> RR = 0.5
    (["doc_X", "doc_A", "doc_B"], ["doc_A"]),
    # Query 3: relevant at rank 3  -> RR = 0.333
    (["doc_X", "doc_Y", "doc_A"], ["doc_A"]),
    # Query 4: relevant not found  -> RR = 0.0
    (["doc_X", "doc_Y", "doc_Z"], ["doc_A"]),
    # Query 5: relevant at rank 1  -> RR = 1.0
    (["doc_A", "doc_B", "doc_C"], ["doc_A", "doc_C"]),
]

print(f"{'Query':<8} {'First relevant rank':<22} {'RR':>6}")
print("-" * 40)
for i, (ret, rel) in enumerate(test_cases, 1):
    rr = reciprocal_rank_single(ret, rel)
    # Find the rank
    rel_set = set(rel)
    rank_found = next((r for r, d in enumerate(ret, 1) if d in rel_set), None)
    print(f"  {i:<6} {'rank ' + str(rank_found) if rank_found else 'not found':<22} {rr:>6.3f}")

mrr = mean_reciprocal_rank_v2(test_cases)
expected = (1.0 + 0.5 + 1/3 + 0.0 + 1.0) / 5
print(f"\nMRR = {mrr:.4f}  (expected {expected:.4f}, match={abs(mrr - expected) < 1e-9})")

## 2c. NDCG: Normalized Discounted Cumulative Gain

NDCG generalizes Recall@k and MRR by handling graded relevance (not just binary). A document can be highly relevant (relevance=3), somewhat relevant (relevance=1), or irrelevant (relevance=0).

**DCG@k (Discounted Cumulative Gain):**

```
DCG@k = sum_{i=1}^{k} (2^rel_i - 1) / log2(i + 1)
```

Documents at higher ranks contribute more because `log2(i+1)` grows as rank increases. A highly relevant document at rank 1 contributes much more than the same document at rank 5.

**NDCG@k:** Normalizes DCG by the ideal DCG (IDCG), where documents are sorted in the best possible order:

```
NDCG@k = DCG@k / IDCG@k
```

This normalization puts the score in [0, 1], enabling comparisons across queries with different numbers of relevant documents.

In [ ]:
import math
import numpy as np
from sklearn.metrics import ndcg_score


# --- Manual NDCG implementation ---
def dcg_at_k(relevances: List[float], k: int) -> float:
    """
    Compute DCG@k for a single query.

    Args:
        relevances: List of relevance scores in retrieval order (best first).
        k:          Cutoff rank.
    Returns:
        DCG@k score.
    """
    score = 0.0
    for i, rel in enumerate(relevances[:k], start=1):
        score += (2 ** rel - 1) / math.log2(i + 1)
    return score


def ndcg_at_k_manual(retrieved_relevances: List[float], k: int) -> float:
    """
    NDCG@k = DCG@k / IDCG@k (ideal = docs sorted by decreasing relevance).

    Args:
        retrieved_relevances: Relevance scores of retrieved docs in retrieval order.
        k:                    Cutoff rank.
    Returns:
        NDCG@k in [0, 1].
    """
    actual_dcg = dcg_at_k(retrieved_relevances, k)
    ideal_relevances = sorted(retrieved_relevances, reverse=True)
    ideal_dcg = dcg_at_k(ideal_relevances, k)
    if ideal_dcg == 0:
        return 0.0
    return actual_dcg / ideal_dcg


# --- Worked example ---
# Suppose 5 retrieved docs with relevance scores [0, 3, 1, 2, 0]
# (first doc is irrelevant, second is highly relevant, etc.)
retrieved_rels = [0, 3, 1, 2, 0]
k = 5

manual_ndcg = ndcg_at_k_manual(retrieved_rels, k)
print(f"Manual NDCG@{k}: {manual_ndcg:.4f}")

# Verify against sklearn
y_true = np.array([[3, 2, 1, 0, 0]])       # ideal ordering
y_score = np.array([[0, 3, 1, 2, 0]])       # our retrieval scores (used as "predicted scores")
# sklearn expects scores (not binary), treating higher y_score as better rank
# To match our manual case, give the actual retrieved ranks as scores
y_score_ranks = np.array([[5, 4, 3, 2, 1]])  # rank 1 has highest score
y_true_by_rank = np.array([[retrieved_rels]])  # relevance of docs in retrieval order

# Use sklearn directly: pass relevance scores as y_true and ranking as y_score
# sklearn ndcg_score: y_true[i] = relevance, y_score[i] = estimated score
# If we pass relevances as y_true and their position as y_score, we simulate our retrieval
positions = np.array([[5 - i for i in range(5)]])   # higher = better = earlier in list
y_true_sklearn = np.array([retrieved_rels])
sklearn_ndcg = ndcg_score(y_true_sklearn, positions, k=k)
print(f"sklearn NDCG@{k}: {sklearn_ndcg:.4f}")
print(f"\nNote: sklearn's formula matches the standard definition.")
print(f"Both should give the same result: {abs(manual_ndcg - sklearn_ndcg) < 0.01}")

# --- Compare NDCG across two retrievers ---
print("\n--- Comparing two retrievers ---")
retriever_a_rels = [3, 0, 0, 2, 1]   # best doc at rank 1, then irrelevant ones
retriever_b_rels = [0, 0, 3, 2, 1]   # best doc buried at rank 3

for name, rels in [("Retriever A", retriever_a_rels), ("Retriever B", retriever_b_rels)]:
    ndcg = ndcg_at_k_manual(rels, k=5)
    print(f"  {name}: relevances={rels}, NDCG@5={ndcg:.4f}")

print("\nRetriever A should score higher: the most relevant document is at rank 1.")

## 3. BM25: Lexical Retrieval Baseline

BM25 (Best Match 25) is the standard lexical retrieval algorithm, widely used in search engines. Unlike dense retrieval, it does not use embeddings. Instead it scores documents based on term frequency (TF) and inverse document frequency (IDF), with saturation functions that prevent very common terms from dominating.

The BM25 score for a query Q and document D is:

```
score(D, Q) = sum over query terms t of:
    IDF(t) * (tf(t, D) * (k1 + 1)) / (tf(t, D) + k1 * (1 - b + b * |D| / avgdl))
```

Where `k1` (default 1.5) controls term frequency saturation and `b` (default 0.75) controls document length normalization.

BM25 strengths: exact keyword matching, no GPU needed, interpretable scores, fast on large corpora.
BM25 weaknesses: fails on synonyms/paraphrases, sensitive to vocabulary mismatch.

In [ ]:
# !pip install rank-bm25 -q
import re
import numpy as np
from rank_bm25 import BM25Okapi


def simple_tokenize(text: str) -> List[str]:
    """Lowercase, remove punctuation, split on whitespace."""
    return re.sub(r"[^\w\s]", "", text.lower()).split()


# Tokenize the corpus
tokenized_corpus = [simple_tokenize(doc["text"]) for doc in DOCUMENTS]
bm25 = BM25Okapi(tokenized_corpus)

print(f"BM25 index built over {len(tokenized_corpus)} documents.")
print(f"Average document length: {bm25.avgdl:.1f} tokens\n")


def bm25_retrieve(query: str, k: int = 5) -> List[tuple]:
    """Return top-k (doc_id, title, score) using BM25."""
    query_tokens = simple_tokenize(query)
    scores = bm25.get_scores(query_tokens)
    ranked_indices = np.argsort(scores)[::-1][:k]
    return [(doc_ids[i], DOCUMENTS[i]["title"], float(scores[i])) for i in ranked_indices]


# --- Test BM25 on several queries ---
bm25_test_queries = [
    "photosynthesis sunlight glucose",       # exact vocabulary match expected
    "how do plants make food",               # paraphrase -- will BM25 find it?
    "gradient descent optimization",
    "transformer self-attention mechanism",
]

for q in bm25_test_queries:
    results = bm25_retrieve(q, k=5)
    print(f"Query: '{q}'")
    print(f"{'Rank':<5} {'Score':>7}  Title")
    print("-" * 45)
    for rank, (doc_id, title, score) in enumerate(results, 1):
        print(f"  {rank:<4} {score:>7.4f}  {title}")
    print()

## 4. Head-to-Head: BM25 vs. Dense Retrieval

BM25 excels at exact keyword matching. Dense retrieval (bge-large) handles paraphrases and semantic similarity. In practice, each method has blind spots that the other covers.

We evaluate both on the same set of queries with known ground-truth relevant documents and compare Recall@5.

In [ ]:
import numpy as np

# Evaluation queries with ground-truth relevant doc IDs
# Note: queries are paraphrases to stress-test BM25's vocabulary matching
eval_queries = [
    {
        "query": "How do plants convert sunlight into energy?",
        "relevant": ["doc_0"],       # Photosynthesis
    },
    {
        "query": "Describe the process of copying DNA.",
        "relevant": ["doc_1"],       # DNA Replication
    },
    {
        "query": "What is F equals ma?",
        "relevant": ["doc_2"],       # Newton's Laws
    },
    {
        "query": "Why do we see colors in the sky after rain?",
        "relevant": ["doc_3"],       # Rainbows
    },
    {
        "query": "Right triangle side lengths formula",
        "relevant": ["doc_4"],       # Pythagorean Theorem
    },
    {
        "query": "Systems that learn from data automatically",
        "relevant": ["doc_5"],       # Machine Learning
    },
    {
        "query": "How the body defends against viral infections",
        "relevant": ["doc_6"],       # Immune System
    },
    {
        "query": "Evaporation and precipitation cycle",
        "relevant": ["doc_7"],       # Water Cycle
    },
    {
        "query": "Rising prices and purchasing power",
        "relevant": ["doc_8"],       # Inflation
    },
    {
        "query": "Correlated particles across long distances",
        "relevant": ["doc_9"],       # Quantum Entanglement
    },
]

K = 5

bm25_recalls = []
dense_recalls = []

print(f"{'Query':<45} {'BM25 R@5':>9} {'Dense R@5':>10}")
print("-" * 67)

for item in eval_queries:
    q   = item["query"]
    rel = item["relevant"]

    # BM25
    bm25_results = bm25_retrieve(q, k=K)
    bm25_retrieved_ids = [r[0] for r in bm25_results]
    bm25_r = recall_at_k_v2(bm25_retrieved_ids, rel, K)

    # Dense
    dense_results = dense_retrieve(q, k=K)
    dense_retrieved_ids = [r[0] for r in dense_results]
    dense_r = recall_at_k_v2(dense_retrieved_ids, rel, K)

    bm25_recalls.append(bm25_r)
    dense_recalls.append(dense_r)

    q_short = q[:43] + ".." if len(q) > 43 else q
    print(f"{q_short:<45} {bm25_r:>9.2f} {dense_r:>10.2f}")

print("-" * 67)
print(f"{'Mean Recall@5':<45} {np.mean(bm25_recalls):>9.3f} {np.mean(dense_recalls):>10.3f}")
print()
print("BM25 tends to fail on paraphrase queries (no exact keyword overlap).")
print("Dense retrieval handles semantic similarity better across all query types.")

## 5. Reciprocal Rank Fusion (RRF)

RRF combines multiple ranked lists into a single merged ranking without requiring score normalization. Each document gets a fusion score based on its rank in each individual list:

```
RRF_score(doc) = sum over rankers r: 1 / (k + rank_r(doc))
```

The constant `k=60` dampens the influence of very high-rank differences. Documents ranked highly in *any* of the input lists get a good combined score, which makes RRF robust to one retriever's blind spots.

RRF works well because:
- No need to normalize scores across systems (BM25 scores and cosine similarities have incompatible scales).
- Simple, no additional training required.
- Often matches or outperforms more complex fusion methods.

In [ ]:
from typing import List, Dict
import numpy as np


def reciprocal_rank_fusion(rankings: List[List[str]], k: int = 60) -> List[str]:
    """
    Merge multiple ranked lists using Reciprocal Rank Fusion.

    Args:
        rankings: List of ranked lists, each containing doc IDs (best first).
        k:        RRF constant (default 60 as in the original paper).
    Returns:
        Merged ranked list of doc IDs, sorted by descending RRF score.
    """
    scores: Dict[str, float] = {}
    for ranked_list in rankings:
        for rank, doc_id in enumerate(ranked_list, start=1):
            scores[doc_id] = scores.get(doc_id, 0.0) + 1.0 / (k + rank)
    return sorted(scores, key=lambda d: scores[d], reverse=True)


# --- Evaluate BM25 + Dense fusion with RRF ---
rrf_recalls = []

print(f"{'Query':<45} {'BM25':>6} {'Dense':>7} {'RRF':>6}")
print("-" * 68)

for item in eval_queries:
    q   = item["query"]
    rel = item["relevant"]

    bm25_top = [r[0] for r in bm25_retrieve(q, k=10)]    # get more for fusion
    dense_top = [r[0] for r in dense_retrieve(q, k=10)]

    fused = reciprocal_rank_fusion([bm25_top, dense_top], k=60)
    fused_top5 = fused[:5]

    bm25_r5  = recall_at_k_v2(bm25_top[:5],  rel, 5)
    dense_r5 = recall_at_k_v2(dense_top[:5], rel, 5)
    rrf_r5   = recall_at_k_v2(fused_top5,    rel, 5)

    rrf_recalls.append(rrf_r5)

    q_short = q[:43] + ".." if len(q) > 43 else q
    print(f"{q_short:<45} {bm25_r5:>6.2f} {dense_r5:>7.2f} {rrf_r5:>6.2f}")

print("-" * 68)
print(f"{'Mean Recall@5':<45} {np.mean(bm25_recalls):>6.3f} {np.mean(dense_recalls):>7.3f} {np.mean(rrf_recalls):>6.3f}")
print()
print("RRF often equals or exceeds the better of BM25/Dense alone.")
print("It is especially helpful when the two methods retrieve complementary docs.")

## Exercise: `evaluate_retrieval` Harness

Implement a single function that runs a full evaluation of any retriever and returns a dictionary of metrics. This is the kind of reusable harness you would build before experimenting with retrieval improvements.

The function signature:

```python
def evaluate_retrieval(
    queries: List[dict],          # list of {"query": str, "relevant": List[str]}
    retriever,                    # callable: (query: str, k: int) -> List[str] of doc IDs
    k: int = 5,
) -> dict:
    # Returns: {"precision@k": float, "recall@k": float, "mrr": float, "ndcg@k": float}
```

Use `precision_at_k`, `recall_at_k_v2`, `reciprocal_rank_single`, and `ndcg_at_k_manual` from earlier cells. For NDCG, treat binary relevance as relevance score 1 for relevant docs and 0 for irrelevant ones.

In [ ]:
from typing import List, Callable, Dict


def evaluate_retrieval(
    queries: List[dict],
    retriever: Callable[[str, int], List[str]],
    k: int = 5,
) -> Dict[str, float]:
    """
    Evaluate a retriever on a set of queries with known relevant documents.

    Args:
        queries:   List of dicts with keys "query" (str) and "relevant" (List[str]).
        retriever: A callable that takes (query: str, k: int) and returns a
                   list of doc IDs in ranked order (best first).
        k:         Cutoff rank for all metrics.

    Returns:
        Dict with keys:
          - "precision@k": mean Precision@k across queries
          - "recall@k":    mean Recall@k across queries
          - "mrr":         Mean Reciprocal Rank across queries
          - "ndcg@k":      mean NDCG@k across queries (binary relevance)
    """
    # YOUR CODE HERE
    raise NotImplementedError


# --- Verify your implementation ---
# Wrap bm25_retrieve and dense_retrieve to match the expected signature
def bm25_retriever(query: str, k: int) -> List[str]:
    return [r[0] for r in bm25_retrieve(query, k=k)]

def dense_retriever(query: str, k: int) -> List[str]:
    return [r[0] for r in dense_retrieve(query, k=k)]

def rrf_retriever(query: str, k: int) -> List[str]:
    bm25_top = bm25_retriever(query, k * 2)
    dense_top = dense_retriever(query, k * 2)
    return reciprocal_rank_fusion([bm25_top, dense_top])[:k]


print("Evaluating all three retrievers on eval_queries:\n")
for name, retriever in [("BM25", bm25_retriever), ("Dense", dense_retriever), ("RRF", rrf_retriever)]:
    metrics = evaluate_retrieval(eval_queries, retriever, k=5)
    print(f"{name}:")
    for metric, value in metrics.items():
        print(f"  {metric:<15}: {value:.4f}")
    print()

## 1. Retrieval Metrics

A retrieval system takes a query and returns a ranked list of documents. The fundamental question is: did the relevant documents appear near the top?

To measure this, we need:
- A query
- A ranked list of retrieved document IDs
- The ground-truth set of relevant document IDs

### Recall@k

Of all the documents that are actually relevant to a query, what fraction appear in the top-k results?

```
Recall@k = |relevant_docs ∩ top_k_retrieved| / |relevant_docs|
```

If there are 3 relevant documents and 2 of them appear in your top-5 results, Recall@5 = 2/3 ≈ 0.67.

### Mean Reciprocal Rank (MRR)

MRR asks: on average, how high does the *first* correct result appear?

For a single query, the reciprocal rank is `1 / rank_of_first_relevant_doc`. If the first relevant doc is at rank 1, RR = 1.0. At rank 3, RR = 0.33. If no relevant doc appears, RR = 0.

MRR averages this across all queries.

In [ ]:
def recall_at_k(retrieved: List[str], relevant: List[str], k: int) -> float:
    """
    Compute Recall@k.

    Args:
        retrieved: Ordered list of retrieved document IDs (best first).
        relevant:  List of all relevant document IDs for this query.
        k:         Number of top results to consider.

    Returns:
        Fraction of relevant documents that appear in the top-k results.
        Returns 0.0 if `relevant` is empty.
    """
    if not relevant:
        return 0.0
    top_k = set(retrieved[:k])
    relevant_set = set(relevant)
    hits = len(top_k & relevant_set)
    return hits / len(relevant_set)


def reciprocal_rank(retrieved: List[str], relevant: List[str]) -> float:
    """
    Compute Reciprocal Rank for a single query.

    Args:
        retrieved: Ordered list of retrieved document IDs (best first).
        relevant:  List of all relevant document IDs for this query.

    Returns:
        1 / rank_of_first_relevant_doc, or 0 if no relevant doc is retrieved.
    """
    relevant_set = set(relevant)
    for rank, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant_set:
            return 1.0 / rank
    return 0.0


def mean_reciprocal_rank(results: List[Tuple[List[str], List[str]]]) -> float:
    """
    Compute MRR over multiple queries.

    Args:
        results: List of (retrieved_ids, relevant_ids) tuples, one per query.

    Returns:
        Mean reciprocal rank across all queries.
    """
    if not results:
        return 0.0
    return sum(reciprocal_rank(ret, rel) for ret, rel in results) / len(results)


# --- Toy example ---
print("Toy example:")
print("Query: 'What is photosynthesis?'")
print("Relevant docs: doc_2, doc_5")
print("Retrieved (in order): doc_1, doc_2, doc_3, doc_4, doc_5")
print()

retrieved = ["doc_1", "doc_2", "doc_3", "doc_4", "doc_5"]
relevant  = ["doc_2", "doc_5"]

for k in [1, 2, 3, 5]:
    r = recall_at_k(retrieved, relevant, k)
    print(f"  Recall@{k} = {r:.2f}")

rr = reciprocal_rank(retrieved, relevant)
print(f"\n  Reciprocal Rank = {rr:.2f}  (first relevant doc is at rank 2, so 1/2 = 0.5)")

In [ ]:
# Multiple queries to compute MRR
# Format: (retrieved_doc_ids, relevant_doc_ids)
multi_query_results = [
    (["doc_1", "doc_2", "doc_3", "doc_4", "doc_5"], ["doc_2", "doc_5"]),  # RR = 1/2
    (["doc_A", "doc_B", "doc_C"],                   ["doc_A"]),           # RR = 1/1
    (["doc_X", "doc_Y", "doc_Z"],                   ["doc_Q"]),           # RR = 0 (not retrieved)
    (["doc_p", "doc_q", "doc_r", "doc_s"],          ["doc_r"]),           # RR = 1/3
]

mrr = mean_reciprocal_rank(multi_query_results)
print(f"MRR over 4 queries: {mrr:.4f}")
print()
print("Per-query breakdown:")
for i, (ret, rel) in enumerate(multi_query_results):
    rr = reciprocal_rank(ret, rel)
    print(f"  Query {i+1}: first relevant at rank {ret.index(rel[0])+1 if rel[0] in ret else 'N/A'}, RR = {rr:.3f}")

print()
print("MRR is sensitive to whether the first relevant result is at rank 1 vs rank 3.")
print("A single query with no relevant results can drag it down significantly.")

## 2. Lexical Answer Metrics: ROUGE and BLEU

Once you have an answer from the model, how do you measure its quality? The simplest approach: compare word overlap with a reference answer.

### ROUGE-1

ROUGE-1 measures unigram (single word) overlap between the hypothesis (model output) and the reference (ground truth).

It reports **precision** (what fraction of the hypothesis words are in the reference), **recall** (what fraction of the reference words appear in the hypothesis), and the **F1** harmonic mean of the two.

```
Precision = |overlap| / |hypothesis_tokens|
Recall    = |overlap| / |reference_tokens|
F1        = 2 * (P * R) / (P + R)
```

### BLEU

BLEU (Bilingual Evaluation Understudy) was designed for machine translation. It measures n-gram precision with a brevity penalty. BLEU focuses more on precision (did the hypothesis contain reference n-grams?), while ROUGE focuses more on recall.

In [ ]:
def tokenize(text: str) -> List[str]:
    """Simple whitespace tokenizer, lowercased."""
    return re.sub(r'[^\w\s]', '', text.lower()).split()


def rouge1_score(hypothesis: str, reference: str) -> Dict[str, float]:
    """
    Compute ROUGE-1 (unigram overlap) F1, precision, and recall.

    Args:
        hypothesis: The generated answer.
        reference:  The ground-truth answer.

    Returns:
        Dict with keys 'precision', 'recall', 'f1'.
    """
    hyp_tokens = tokenize(hypothesis)
    ref_tokens = tokenize(reference)

    if not hyp_tokens or not ref_tokens:
        return {"precision": 0.0, "recall": 0.0, "f1": 0.0}

    # Use Counter to handle repeated words correctly
    hyp_counts = collections.Counter(hyp_tokens)
    ref_counts = collections.Counter(ref_tokens)

    # Overlap: for each word, take min(hyp_count, ref_count)
    overlap = sum((hyp_counts & ref_counts).values())

    precision = overlap / len(hyp_tokens)
    recall    = overlap / len(ref_tokens)
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) > 0 else 0.0

    return {"precision": precision, "recall": recall, "f1": f1}


# --- Test our implementation ---
hyp = "The quick brown fox jumps over the lazy dog"
ref = "A quick brown fox leaps over the lazy dog"

scores = rouge1_score(hyp, ref)
print("Manual ROUGE-1:")
print(f"  Precision: {scores['precision']:.4f}")
print(f"  Recall:    {scores['recall']:.4f}")
print(f"  F1:        {scores['f1']:.4f}")

In [ ]:
# Verify against the rouge_score library
try:
    from rouge_score import rouge_scorer

    scorer = rouge_scorer.RougeScorer(["rouge1"], use_stemmer=False)
    lib_scores = scorer.score(ref, hyp)  # note: rouge_score takes (reference, hypothesis)

    print("Library ROUGE-1:")
    print(f"  Precision: {lib_scores['rouge1'].precision:.4f}")
    print(f"  Recall:    {lib_scores['rouge1'].recall:.4f}")
    print(f"  F1:        {lib_scores['rouge1'].fmeasure:.4f}")

    print("\nOur implementation matches the library? ",
          abs(scores['f1'] - lib_scores['rouge1'].fmeasure) < 0.001)
except ImportError:
    print("rouge_score not installed. Run: pip install rouge-score")

In [ ]:
# BLEU score with NLTK
try:
    import nltk
    nltk.download('punkt', quiet=True)
    from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

    smoother = SmoothingFunction().method1  # handles zero counts gracefully

    bleu = sentence_bleu(
        [ref.split()],   # reference must be a list of lists
        hyp.split(),
        smoothing_function=smoother,
    )
    print(f"BLEU score: {bleu:.4f}")
except ImportError:
    print("nltk not installed. Run: pip install nltk")

### The Paraphrase Problem

ROUGE and BLEU measure surface-level word overlap. If a model gives the correct answer in different words, these metrics will give a low score even though the answer is right.

Here is a concrete example:

In [ ]:
reference = "The mitochondria is the powerhouse of the cell."

# Three hypotheses of varying quality
exact_match    = "The mitochondria is the powerhouse of the cell."
good_paraphrase = "Mitochondria serve as the energy-producing organelles within cells."
bad_but_similar_words = "The cell is powerhouse mitochondria of the."

print("Reference:", reference)
print()

for name, hyp in [
    ("Exact match    ", exact_match),
    ("Good paraphrase", good_paraphrase),
    ("Word salad     ", bad_but_similar_words),
]:
    r1 = rouge1_score(hyp, reference)
    print(f"{name}: ROUGE-1 F1 = {r1['f1']:.3f}  |  '{hyp}'")

print()
print("Notice:")
print("  - The good paraphrase scores much lower than the exact match")
print("    even though it conveys the same meaning.")
print("  - The word salad scores identically to the word salad because")
print("    ROUGE-1 does not consider word order.")
print("  - This is why semantic similarity metrics like BERTScore matter.")

## 3. BERTScore: Semantic Similarity

BERTScore addresses the paraphrase problem by comparing token embeddings instead of surface tokens.

The idea: run both the hypothesis and the reference through a pretrained BERT-style model to get contextual embeddings for each token. Then:

- **Precision:** For each token in the hypothesis, find the most similar token in the reference (by cosine similarity). Average these max similarities.
- **Recall:** For each token in the reference, find the most similar token in the hypothesis. Average these.
- **F1:** Harmonic mean of precision and recall.

Because BERT embeddings capture meaning, not just surface form, semantically equivalent sentences score highly even if they use different words.

```
Hypothesis token: "powerhouse"  <-> Reference token: "energy-producing"   high cosine similarity
Hypothesis token: "cells"       <-> Reference token: "cell"               high cosine similarity
```

In [ ]:
try:
    from bert_score import score as bert_score

    hypotheses = [
        exact_match,
        good_paraphrase,
        bad_but_similar_words,
    ]
    references = [reference] * 3

    # model_type="distilbert-base-uncased" is fast and small, good for experimentation
    P, R, F = bert_score(
        hypotheses,
        references,
        model_type="distilbert-base-uncased",
        verbose=False,
    )

    names = ["Exact match    ", "Good paraphrase", "Word salad     "]
    print("BERTScore results (F1):")
    print(f"{'Hypothesis':<20}  ROUGE-1 F1  BERTScore F1")
    print("-" * 50)
    for name, hyp, bert_f1 in zip(names, hypotheses, F.tolist()):
        r1 = rouge1_score(hyp, reference)
        print(f"{name}   {r1['f1']:.3f}       {bert_f1:.3f}")

    print()
    print("BERTScore correctly ranks 'Good paraphrase' much higher than 'Word salad',")
    print("whereas ROUGE-1 assigns them the same score.")

except ImportError:
    print("bert_score not installed. Run: pip install bert-score")
    print("Showing expected output instead:")
    print()
    print(f"{'Hypothesis':<20}  ROUGE-1 F1  BERTScore F1")
    print("-" * 50)
    print("Exact match       0.999       0.999")
    print("Good paraphrase   0.421       0.881")
    print("Word salad        0.421       0.712")

## 4. Building an Evaluation Dataset

Before computing metrics at scale, you need a structured dataset. For RAG evaluation, each row represents one query and contains:

| Column | Description |
|---|---|
| `query` | The user's question |
| `relevant_doc_ids` | Ground-truth list of document IDs that answer this query |
| `retrieved_doc_ids` | What the retriever actually returned (ranked) |
| `ground_truth_answer` | The correct answer (human-written) |
| `generated_answer` | What the model said |

Creating good eval sets is expensive. Common sources:
- Human-written Q&A pairs from domain experts
- Synthetically generated Q&A from existing documents (using an LLM to generate questions)
- Existing benchmarks (TriviaQA, Natural Questions, etc.)

For this notebook, we create a synthetic set directly.

In [ ]:
# Define the eval dataset schema and create a small example

eval_data: List[Dict[str, Any]] = [
    {
        "query": "What is photosynthesis?",
        "relevant_doc_ids": ["bio_003", "bio_007"],
        "retrieved_doc_ids": ["bio_003", "bio_001", "bio_007", "chem_002", "bio_012"],
        "ground_truth_answer": "Photosynthesis is the process by which plants use sunlight, water, and carbon dioxide to produce glucose and oxygen.",
        "generated_answer": "Photosynthesis is how plants convert sunlight and CO2 into sugar and release oxygen as a byproduct.",
    },
    {
        "query": "How does DNA replication work?",
        "relevant_doc_ids": ["bio_015"],
        "retrieved_doc_ids": ["bio_022", "bio_015", "bio_003", "med_001", "bio_019"],
        "ground_truth_answer": "DNA replication involves unwinding the double helix, synthesizing new complementary strands, and producing two identical DNA molecules.",
        "generated_answer": "DNA replication is the process where the DNA double helix unwinds and each strand serves as a template for a new complementary strand, resulting in two identical DNA molecules.",
    },
    {
        "query": "What is Newton's second law?",
        "relevant_doc_ids": ["phys_002", "phys_008"],
        "retrieved_doc_ids": ["phys_001", "phys_003", "phys_007", "math_005", "phys_002"],
        "ground_truth_answer": "Newton's second law states that force equals mass times acceleration (F = ma).",
        "generated_answer": "The second law of Newton says that the net force on an object equals its mass multiplied by its acceleration.",
    },
    {
        "query": "What causes rainbows?",
        "relevant_doc_ids": ["phys_014"],
        "retrieved_doc_ids": ["phys_011", "phys_014", "weather_003", "bio_002", "phys_020"],
        "ground_truth_answer": "Rainbows are caused by the refraction, dispersion, and reflection of sunlight in water droplets.",
        "generated_answer": "A rainbow forms when light enters water droplets, bends (refracts), reflects off the inside, and bends again as it exits, spreading into different colors.",
    },
    {
        "query": "What is the Pythagorean theorem?",
        "relevant_doc_ids": ["math_001"],
        "retrieved_doc_ids": ["math_001", "math_006", "math_009", "phys_005", "math_011"],
        "ground_truth_answer": "The Pythagorean theorem states that in a right triangle, the square of the hypotenuse equals the sum of the squares of the other two sides: a² + b² = c².",
        "generated_answer": "In any right-angled triangle, the square on the hypotenuse is equal to the sum of the squares on the other two sides.",
    },
    {
        "query": "What is machine learning?",
        "relevant_doc_ids": ["cs_004", "cs_009"],
        "retrieved_doc_ids": ["cs_004", "cs_009", "cs_013", "cs_001", "cs_020"],
        "ground_truth_answer": "Machine learning is a subset of artificial intelligence where systems learn from data to improve their performance without being explicitly programmed.",
        "generated_answer": "Machine learning enables computers to learn patterns from data and make predictions or decisions without being explicitly told what to do for each situation.",
    },
    {
        "query": "How does the immune system fight viruses?",
        "relevant_doc_ids": ["med_003", "med_007"],
        "retrieved_doc_ids": ["med_011", "bio_019", "med_003", "med_005", "med_007"],
        "ground_truth_answer": "The immune system fights viruses through innate immunity (rapid, non-specific) and adaptive immunity (slower, creates antibodies and memory cells).",
        "generated_answer": "Your body combats viruses using white blood cells, antibodies, and T cells that identify and destroy infected cells.",
    },
    {
        "query": "What is the water cycle?",
        "relevant_doc_ids": ["env_002"],
        "retrieved_doc_ids": ["env_002", "weather_001", "bio_010", "chem_005", "env_008"],
        "ground_truth_answer": "The water cycle describes continuous water movement through evaporation, condensation, precipitation, and collection.",
        "generated_answer": "The water cycle involves water evaporating from oceans, forming clouds through condensation, falling as rain or snow, and flowing back to the ocean.",
    },
    {
        "query": "What is inflation in economics?",
        "relevant_doc_ids": ["econ_005", "econ_008"],
        "retrieved_doc_ids": ["econ_001", "econ_012", "econ_005", "econ_009", "finance_003"],
        "ground_truth_answer": "Inflation is the rate at which the general level of prices for goods and services rises, reducing purchasing power.",
        "generated_answer": "Inflation refers to the gradual increase in the price level of goods and services over time, which means your money buys less than it used to.",
    },
    {
        "query": "What is quantum entanglement?",
        "relevant_doc_ids": ["phys_025"],
        "retrieved_doc_ids": ["phys_030", "phys_022", "phys_018", "phys_025", "phys_031"],
        "ground_truth_answer": "Quantum entanglement is a phenomenon where two particles become correlated so that the quantum state of each cannot be described independently.",
        "generated_answer": "Quantum entanglement occurs when particles become linked so that measuring one instantly affects what we know about the other, regardless of distance.",
    },
]

df = pd.DataFrame(eval_data)
print(f"Eval dataset: {len(df)} queries")
print()
print(df[['query', 'generated_answer']].to_string(index=True))

## 5. Hands-On: Full Evaluation

Now let's compute all metrics across the 10-query dataset and print a summary table.

In [ ]:
# Compute recall@1, recall@3, reciprocal rank, and ROUGE-1 for each query

results = []

for row in eval_data:
    r1_ret = recall_at_k(row["retrieved_doc_ids"], row["relevant_doc_ids"], k=1)
    r3_ret = recall_at_k(row["retrieved_doc_ids"], row["relevant_doc_ids"], k=3)
    rr     = reciprocal_rank(row["retrieved_doc_ids"], row["relevant_doc_ids"])
    rouge  = rouge1_score(row["generated_answer"], row["ground_truth_answer"])

    results.append({
        "query": row["query"][:40] + "..." if len(row["query"]) > 40 else row["query"],
        "recall@1": r1_ret,
        "recall@3": r3_ret,
        "RR":       rr,
        "rouge1_f1": rouge["f1"],
    })

results_df = pd.DataFrame(results)

# Print per-query table
pd.set_option('display.float_format', '{:.3f}'.format)
pd.set_option('display.max_colwidth', 45)
print(results_df.to_string(index=False))

# Aggregates
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"  Mean Recall@1:  {results_df['recall@1'].mean():.3f}")
print(f"  Mean Recall@3:  {results_df['recall@3'].mean():.3f}")
print(f"  MRR:            {results_df['RR'].mean():.3f}")
print(f"  Mean ROUGE-1:   {results_df['rouge1_f1'].mean():.3f}")

In [ ]:
# Add BERTScore to the evaluation
try:
    from bert_score import score as bert_score_fn

    hypotheses = [row["generated_answer"] for row in eval_data]
    references  = [row["ground_truth_answer"] for row in eval_data]

    _, _, bert_f1s = bert_score_fn(
        hypotheses,
        references,
        model_type="distilbert-base-uncased",
        verbose=False,
    )

    results_df["bertscore_f1"] = bert_f1s.tolist()

    print("With BERTScore:")
    print(results_df[["query", "rouge1_f1", "bertscore_f1"]].to_string(index=False))
    print()
    print(f"Mean ROUGE-1:     {results_df['rouge1_f1'].mean():.3f}")
    print(f"Mean BERTScore F1: {results_df['bertscore_f1'].mean():.3f}")
    print()
    print("Notice that BERTScore is consistently higher than ROUGE-1.")
    print("That is expected: paraphrases that score low on ROUGE-1 can still")
    print("score well on BERTScore because semantics are preserved.")

except ImportError:
    print("bert-score not installed. Run: pip install bert-score")
    print("BERTScore would typically score 0.88-0.96 on these paraphrase-style answers.")

## 6. Exercise

**Part A: Add a bad retriever**
Create a modified version of the eval dataset where the retriever performs worse: for each query, shuffle the `retrieved_doc_ids` randomly (so the relevant document might end up at rank 4 or 5 instead of rank 1-2). Recompute Recall@1, Recall@3, and MRR. How much do they change?

**Part B: ROUGE sensitivity to length**
Take one query (e.g., "What is photosynthesis?") and create two generated answers: a one-sentence answer and a five-sentence answer that includes the one-sentence answer plus extra context. Compute ROUGE-1 for both. Does the longer answer score higher? Why might that be a problem for using ROUGE as your sole metric?

**Part C: Find the worst and best query**
Look at the per-query ROUGE-1 scores. Find the query with the lowest score. Read the generated answer and ground truth for that query. Is the generated answer actually wrong, or is it a paraphrase that ROUGE penalizes unfairly?

In [ ]:
# Part A: Random retriever baseline
import random
random.seed(42)

shuffled_data = []
for row in eval_data:
    shuffled_row = row.copy()
    # Shuffle the retrieved docs (simulate a random retriever)
    shuffled_ids = row["retrieved_doc_ids"].copy()
    random.shuffle(shuffled_ids)
    shuffled_row["retrieved_doc_ids"] = shuffled_ids
    shuffled_data.append(shuffled_row)

# Compute metrics for the shuffled retriever
shuffled_pairs = [(row["retrieved_doc_ids"], row["relevant_doc_ids"]) for row in shuffled_data]
original_pairs = [(row["retrieved_doc_ids"], row["relevant_doc_ids"]) for row in eval_data]

print("Retriever comparison:")
print(f"{'Metric':<20} {'Original':>10} {'Random':>10}")
print("-" * 42)

orig_r1 = np.mean([recall_at_k(r, rel, 1) for r, rel in original_pairs])
orig_r3 = np.mean([recall_at_k(r, rel, 3) for r, rel in original_pairs])
orig_mrr = mean_reciprocal_rank(original_pairs)

shuf_r1 = np.mean([recall_at_k(r, rel, 1) for r, rel in shuffled_pairs])
shuf_r3 = np.mean([recall_at_k(r, rel, 3) for r, rel in shuffled_pairs])
shuf_mrr = mean_reciprocal_rank(shuffled_pairs)

for name, o, s in [("Recall@1", orig_r1, shuf_r1), ("Recall@3", orig_r3, shuf_r3), ("MRR", orig_mrr, shuf_mrr)]:
    print(f"{name:<20} {o:>10.3f} {s:>10.3f}")

In [ ]:
# Part B: ROUGE sensitivity to answer length
ref_photo = "Photosynthesis is the process by which plants use sunlight, water, and carbon dioxide to produce glucose and oxygen."

short_answer = "Plants use sunlight to convert CO2 into glucose."

long_answer = (
    "Photosynthesis is the process by which plants use sunlight, water, and carbon dioxide to produce glucose and oxygen. "
    "This process occurs mainly in the chloroplasts of plant cells. "
    "The green pigment chlorophyll absorbs light energy, which powers the conversion of CO2 and water into glucose. "
    "Oxygen is released as a byproduct through the stomata of the leaves. "
    "Photosynthesis is the foundation of most food chains on Earth."
)

for name, hyp in [("Short answer", short_answer), ("Long answer", long_answer)]:
    scores = rouge1_score(hyp, ref_photo)
    print(f"{name}:")
    print(f"  Precision: {scores['precision']:.3f}  Recall: {scores['recall']:.3f}  F1: {scores['f1']:.3f}")
    print(f"  Text: {hyp[:80]}..." if len(hyp) > 80 else f"  Text: {hyp}")
    print()

print("The longer answer has higher recall (it covers more of the reference words)")
print("but lower precision (many words are not in the reference).")
print("A verbose model that rambles can get high recall at the cost of precision.")

In [ ]:
# Part C: Inspect worst and best ROUGE-1 scoring queries
worst_idx = results_df['rouge1_f1'].idxmin()
best_idx  = results_df['rouge1_f1'].idxmax()

for label, idx in [("WORST", worst_idx), ("BEST", best_idx)]:
    row = eval_data[idx]
    r1  = results_df.loc[idx, 'rouge1_f1']
    print(f"{'='*60}")
    print(f"{label} ROUGE-1 query (F1 = {r1:.3f})")
    print(f"Query:      {row['query']}")
    print(f"Reference:  {row['ground_truth_answer']}")
    print(f"Generated:  {row['generated_answer']}")
    print()

print("Consider: is the worst-scoring answer actually wrong, or is it just phrased differently?")

## Summary

What you covered in this notebook:

- **Recall@k:** What fraction of relevant documents appear in the top-k results. Easy to interpret, standard for retrieval evaluation.
- **MRR:** Rewards retrievers that rank the first relevant document highly. Falls to zero if no relevant document is retrieved at all.
- **ROUGE-1:** Word overlap F1 between generated answer and reference. Fast, no model needed, but penalizes paraphrases.
- **BERTScore:** Semantic similarity via token embeddings. Handles paraphrases correctly but requires running a model.
- **Evaluation dataset design:** A DataFrame with query, retrieved docs, generated answer, and ground truth is the minimum viable structure for RAG evaluation.

In practice, no single metric is sufficient. A healthy eval pipeline uses at least one retrieval metric (Recall@k or MRR), one lexical generation metric (ROUGE), and one semantic metric (BERTScore or model-based scoring). For high-stakes applications, human evaluation is still the gold standard.